# NB08 — Ablation Study

Two ablations on the main random split (pure 20% test), single encoder (default **BanglaBERT**, 1 seed,
6 epochs to keep it tractable — relative deltas are what matter):

**A. Component ablation (additive):** base → +focal+CW → +balanced-sampler → +multi-sample-dropout →
+R-Drop → +FGM → +EMA (full method). Isolates each component's contribution.

**B. Taxonomy ablation:** 5-class vs 9-class (old consolidation), full method — justifies the 5-class choice.

Same training machinery as NB05. Each config = one run; results → `outputs/ablation/`.


In [ ]:
import os, json, time, math, random, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, roc_auc_score
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available())

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
DEBUG = True; DEBUG_SAMPLES = 400
ABLATION_MODEL = "banglabert"
MODEL_PATH = {"banglabert":"csebuetnlp/banglabert","muril":"google/muril-base-cased","xlmr":"xlm-roberta-base"}[ABLATION_MODEL]
SEED = 42
BASE = {"max_length":128,"batch_size":16,"eval_batch_size":32,"grad_accum_steps":2,"num_workers":0,
        "epochs":6,"encoder_lr":2e-5,"head_lr":8e-5,"weight_decay":0.01,"warmup_ratio":0.10,
        "max_grad_norm":1.0,"label_smoothing":0.03,"focal_gamma":2.0,"class_weight_beta":0.999,
        "dropout":0.25,"head_hidden_dim":384,"patience":3,"use_fp16":torch.cuda.is_available(),
        "sampler_alpha":0.5,"fgm_epsilon":1.0,"rdrop_alpha":1.0,"ema_decay":0.999,
        # toggles (ablation flips these)
        "use_focal":False,"use_cw":False,"use_balanced_sampler":False,"n_msd":1,
        "use_rdrop":False,"use_fgm":False,"use_ema":False}
SPLIT_DIR="../data/splits"; OUT="../outputs/ablation"; os.makedirs(OUT,exist_ok=True)
TEXT="text_clean"; FORCE=False
if DEBUG: BASE["epochs"]=2; print("⚠ DEBUG")

In [ ]:
# ── Training machinery (same as NB05) ───────────────────────────────────────
class DS(Dataset):
    def __init__(self,df,tok,maxlen,enc,label):
        self.t=df.reset_index(drop=True)[TEXT].fillna("").astype(str).tolist()
        self.y=[int(enc.get(v,-1)) for v in df[label]]; self.tok,self.ml=tok,maxlen
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(self.t[i],max_length=self.ml,truncation=True,padding=False)
        it={k:torch.tensor(v,dtype=torch.long) for k,v in e.items()}; it["label"]=torch.tensor(self.y[i],dtype=torch.long); return it
class Collator:
    def __init__(self,tok): self.tok=tok
    def __call__(self,fs):
        lb=torch.stack([f["label"] for f in fs]); ft=[{k:v for k,v in f.items() if k!="label"} for f in fs]
        b=self.tok.pad(ft,padding=True,return_tensors="pt"); b["label"]=lb; return b
class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,weight=None,ls=0.0): super().__init__(); self.g,self.w,self.ls=gamma,weight,ls
    def forward(self,lg,t):
        ce=F.cross_entropy(lg,t,weight=self.w,reduction="none",label_smoothing=self.ls); return (((1-torch.exp(-ce))**self.g)*ce).mean()
def build_cw(s,enc,beta=0.999,mw=10.0):
    m=s.map(enc).dropna().astype(int); n=len(enc); c=m.value_counts().sort_index(); w=np.ones(n,dtype=np.float32)
    for i in range(n):
        k=int(c.get(i,0))
        if k>0: w[i]=1.0/max((1.0-beta**k)/max(1.0-beta,1e-12),1e-9)
    w=np.clip(w,w.min(),w.min()*mw); w=w/w.mean(); return torch.tensor(w,dtype=torch.float32,device=device)
def make_sampler(df,enc,label,alpha=0.5):
    y=df[label].map(enc).fillna(-1).astype(int).values; cc=np.bincount(y[y>=0],minlength=len(enc)).astype(float); cc[cc==0]=1.0
    cw=(1.0/cc)**float(alpha); sw=np.where(y>=0,cw[np.clip(y,0,None)],0.0)
    return WeightedRandomSampler(torch.as_tensor(sw,dtype=torch.double),len(sw),replacement=True)
class MSDHead(nn.Module):
    def __init__(self,h,n,dp=0.25,inner=384,nm=4):
        super().__init__(); inner=min(inner,h); self.pre=nn.Sequential(nn.Linear(h,inner),nn.GELU(),nn.LayerNorm(inner))
        self.drops=nn.ModuleList([nn.Dropout(dp) for _ in range(max(1,nm))]); self.out=nn.Linear(inner,n)
    def forward(self,x):
        h=self.pre(x)
        if self.training and len(self.drops)>1: return torch.stack([self.out(d(h)) for d in self.drops],0).mean(0)
        return self.out(self.drops[0](h))
class Encoder(nn.Module):
    def __init__(self,name,n,dp=0.25,inner=384,nm=4):
        super().__init__(); self.encoder=AutoModel.from_pretrained(name); h=self.encoder.config.hidden_size
        self._tti=self.encoder.config.model_type.lower() not in ("roberta","xlm-roberta"); self.head=MSDHead(h,n,dp,inner,nm)
    def _pool(self,o,m):
        hs=o.last_hidden_state; return 0.5*hs[:,0]+0.5*((hs*m.unsqueeze(-1).float()).sum(1)/m.sum(1,keepdim=True).float().clamp(1))
    def forward(self,ids,mask,tti=None):
        kw={"input_ids":ids,"attention_mask":mask}
        if self._tti and tti is not None: kw["token_type_ids"]=tti
        return self.head(self._pool(self.encoder(**kw),mask))
def uparams(model,e,h,wd):
    nd=["bias","LayerNorm.weight","LayerNorm.bias"]; g=[]
    for grp,lr in [(model.encoder,e),(model.head,h)]:
        d,nde=[],[]
        for n,p in grp.named_parameters():
            if p.requires_grad: (nde if any(x in n for x in nd) else d).append(p)
        g+=[{"params":d,"lr":lr,"weight_decay":wd},{"params":nde,"lr":lr,"weight_decay":0.0}]
    return g
class FGM:
    def __init__(self,m,eps=1.0,key="word_embeddings"): self.m,self.e,self.k,self.b=m,eps,key,{}
    def attack(self):
        for n,p in self.m.named_parameters():
            if p.requires_grad and self.k in n and p.grad is not None:
                self.b[n]=p.data.clone(); nm=torch.norm(p.grad)
                if nm!=0 and not torch.isnan(nm): p.data.add_(self.e*p.grad/nm)
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.b: p.data=self.b[n]
        self.b={}
class EMA:
    def __init__(self,m,d=0.999): self.d=d; self.s={n:p.detach().clone() for n,p in m.named_parameters() if p.requires_grad}; self.b={}
    def update(self,m):
        for n,p in m.named_parameters():
            if p.requires_grad: self.s[n].mul_(self.d).add_(p.detach(),alpha=1-self.d)
    def apply_shadow(self,m):
        self.b={n:p.detach().clone() for n,p in m.named_parameters() if p.requires_grad}
        for n,p in m.named_parameters():
            if p.requires_grad: p.data.copy_(self.s[n])
    def restore(self,m):
        for n,p in m.named_parameters():
            if p.requires_grad and n in self.b: p.data.copy_(self.b[n])
        self.b={}
def rdrop_kl(lp,lq):
    pl,ql=F.log_softmax(lp,-1),F.log_softmax(lq,-1); p,q=pl.exp(),ql.exp()
    return 0.5*((p*(pl-ql)).sum(-1)+(q*(ql-pl)).sum(-1)).mean()
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
@torch.no_grad()
def evaluate(model,loader,nc):
    model.eval(); P,Y,PR=[],[],[]
    for b in loader:
        b={k:v.to(device) for k,v in b.items()}
        pr=F.softmax(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")),-1).cpu().numpy()
        y=b["label"].cpu().numpy(); m=y>=0; P.extend(pr[m].argmax(-1)); Y.extend(y[m]); PR.extend(pr[m])
    y,p,pr=np.array(Y),np.array(P),np.array(PR)
    rec={"macro_f1":round(float(f1_score(y,p,average="macro",zero_division=0)),4),
         "weighted_f1":round(float(f1_score(y,p,average="weighted",zero_division=0)),4),
         "accuracy":round(float(accuracy_score(y,p)),4),"mcc":round(float(matthews_corrcoef(y,p)),4)}
    try: rec["auroc"]=round(float(roc_auc_score(y,pr,multi_class="ovr",average="macro",labels=list(range(nc)))),4)
    except Exception: rec["auroc"]=float("nan")
    return rec
print("machinery ok")

In [ ]:
# ── One ablation run ────────────────────────────────────────────────────────
def run_config(cfg, label_col, train_df, val_df, test_df, enc, nc, tag):
    set_seed(SEED)
    tok=AutoTokenizer.from_pretrained(MODEL_PATH); coll=Collator(tok)
    lkw=dict(collate_fn=coll,num_workers=cfg["num_workers"],pin_memory=torch.cuda.is_available())
    ds=DS(train_df,tok,cfg["max_length"],enc,label_col)
    if cfg["use_balanced_sampler"]:
        tl=DataLoader(ds,batch_size=cfg["batch_size"],sampler=make_sampler(train_df,enc,label_col,cfg["sampler_alpha"]),shuffle=False,drop_last=True,**lkw)
    else:
        tl=DataLoader(ds,batch_size=cfg["batch_size"],shuffle=True,drop_last=True,**lkw)
    vl=DataLoader(DS(val_df,tok,cfg["max_length"],enc,label_col),batch_size=cfg["eval_batch_size"],shuffle=False,**lkw)
    te=DataLoader(DS(test_df,tok,cfg["max_length"],enc,label_col),batch_size=cfg["eval_batch_size"],shuffle=False,**lkw)
    model=Encoder(MODEL_PATH,nc,cfg["dropout"],cfg["head_hidden_dim"],cfg["n_msd"]).to(device)
    cw=build_cw(train_df[label_col],enc,cfg["class_weight_beta"]) if cfg["use_cw"] else None
    crit=FocalLoss(cfg["focal_gamma"],cw,cfg["label_smoothing"]) if cfg["use_focal"] \
         else (lambda lg,t: F.cross_entropy(lg,t,weight=cw,label_smoothing=cfg["label_smoothing"]))
    opt=torch.optim.AdamW(uparams(model,cfg["encoder_lr"],cfg["head_lr"],cfg["weight_decay"]))
    ns=math.ceil(len(tl)/cfg["grad_accum_steps"])*cfg["epochs"]; sch=get_linear_schedule_with_warmup(opt,int(ns*cfg["warmup_ratio"]),ns)
    scaler=torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type=="cuda") else None
    fgm=FGM(model,cfg["fgm_epsilon"]) if cfg["use_fgm"] else None
    ema=EMA(model,cfg["ema_decay"]) if cfg["use_ema"] else None
    best,noimp,t0=-1.0,0,time.time()
    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for step,b in enumerate(tl,1):
            b={k:v.to(device) for k,v in b.items()}; y=b["label"]
            with torch.autocast(device_type=device.type,enabled=scaler is not None):
                l1=model(b["input_ids"],b["attention_mask"],b.get("token_type_ids"))
                if cfg["use_rdrop"]:
                    l2=model(b["input_ids"],b["attention_mask"],b.get("token_type_ids"))
                    loss=0.5*(crit(l1,y)+crit(l2,y))+cfg["rdrop_alpha"]*rdrop_kl(l1,l2)
                else: loss=crit(l1,y)
                loss=loss/cfg["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type,enabled=scaler is not None):
                    la=crit(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")),y)/cfg["grad_accum_steps"]
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            if step%cfg["grad_accum_steps"]==0 or step==len(tl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),cfg["max_grad_norm"])
                (scaler.step(opt),scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
        if ema is not None: ema.apply_shadow(model)
        vm=evaluate(model,vl,nc)["macro_f1"]
        if vm>best: best,noimp=vm,0; torch.save(model.state_dict(),f"{OUT}/_tmp_best.pt")
        else: noimp+=1
        if ema is not None: ema.restore(model)
        if noimp>=cfg["patience"]: break
    model.load_state_dict(torch.load(f"{OUT}/_tmp_best.pt",map_location=device,weights_only=True))
    m=evaluate(model,te,nc); m["config"]=tag; m["minutes"]=round((time.time()-t0)/60,1)
    del model; torch.cuda.empty_cache()
    print(f"  {tag:18s} macroF1={m['macro_f1']} wF1={m['weighted_f1']} acc={m['accuracy']} mcc={m['mcc']} [{m['minutes']}min]")
    return m
print("runner ok")

In [ ]:
# ── A. Component ablation (5-class, additive) ───────────────────────────────
train_df=pd.read_csv(f"{SPLIT_DIR}/random_train.csv"); val_df=pd.read_csv(f"{SPLIT_DIR}/random_val.csv"); test_df=pd.read_csv(f"{SPLIT_DIR}/random_test.csv")
if DEBUG:
    train_df=train_df.groupby("label5",group_keys=False).apply(lambda g:g.sample(min(len(g),DEBUG_SAMPLES//5),random_state=42))
    val_df=val_df.sample(min(len(val_df),300),random_state=42); test_df=test_df.sample(min(len(test_df),300),random_state=42)
classes5=sorted(train_df["label5"].unique()); enc5={c:i for i,c in enumerate(classes5)}; print("5 classes:",enc5)

steps=[("base",{}),
       ("+focal+CW",{"use_focal":True,"use_cw":True}),
       ("+sampler",{"use_balanced_sampler":True}),
       ("+MSD",{"n_msd":4}),
       ("+RDrop",{"use_rdrop":True}),
       ("+FGM",{"use_fgm":True}),
       ("+EMA(full)",{"use_ema":True})]
cfg=dict(BASE); ablation=[]
print("\nA. COMPONENT ABLATION (5-class, BanglaBERT)")
for name,delta in steps:
    cfg.update(delta)
    ablation.append(run_config(dict(cfg),"label5",train_df,val_df,test_df,enc5,5,name))
pd.DataFrame(ablation).to_csv(f"{OUT}/component_ablation.csv",index=False)
print(f"✅ saved {OUT}/component_ablation.csv")

In [ ]:
# ── B. Taxonomy ablation: 5-class vs 9-class (full method) ──────────────────
# Derive the OLD 9-class label from raw label_type for the same rows.
import re
NINE_MAP={"none":"none","not bully":"none","sexual":"sexual","gender":"gender","religious":"religious",
          "religion":"religious","threat":"threat","calltoviolence":"threat","abusive/violence":"abusive",
          "abusive":"abusive","violence":"abusive","personal offense":"personal","personal":"personal",
          "political":"political","troll":"abusive","slander":"other","spam":"other","origin":"other",
          "body shaming":"other","misc":"other"}
NINE_PRIO=["threat","sexual","religious","gender","political","abusive","personal","other","none"]
def to9(raw):
    if not isinstance(raw,str) or not raw.strip(): return "none"
    parts=[p.strip().lower() for p in re.split(r"[,_]",raw.strip()) if p.strip()]; b=set()
    for p in parts:
        if p in NINE_MAP: b.add(NINE_MAP[p])
        else:
            for k,v in NINE_MAP.items():
                if k in p: b.add(v); break
    if not b: return "other"
    for c in NINE_PRIO:
        if c in b: return c
    return "none"
for d in (train_df,val_df,test_df):
    d["label9"]=d["label_type"].apply(to9)
    d.loc[d["label_binary"]==0,"label9"]="none"
classes9=sorted(set(train_df["label9"])|set(val_df["label9"])|set(test_df["label9"])); enc9={c:i for i,c in enumerate(classes9)}
full=dict(BASE); full.update({"use_focal":True,"use_cw":True,"use_balanced_sampler":True,"n_msd":4,"use_rdrop":True,"use_fgm":True,"use_ema":True})
print("\nB. TAXONOMY ABLATION (full method)")
tax=[run_config(dict(full),"label5",train_df,val_df,test_df,enc5,5,"5-class(full)"),
     run_config(dict(full),"label9",train_df,val_df,test_df,enc9,len(classes9),"9-class(full)")]
pd.DataFrame(tax).to_csv(f"{OUT}/taxonomy_ablation.csv",index=False)
print(f"✅ saved {OUT}/taxonomy_ablation.csv")
print("\nSummary (5 vs 9):"); print(pd.DataFrame(tax)[["config","macro_f1","weighted_f1","accuracy","mcc"]].to_string(index=False))